# Lahore Societies Map + Average Prices (Folium)

This notebook scans society maps in `../societies`, aggregates average listing prices from `../data` (Graana + Zameen), and renders an interactive Folium map saved to `../dha_price_heatmap.html`.

Steps:
- Discover shapefiles in `../societies/**/*.shp`
- Load scraped CSVs, parse prices to PKR, and compute per-society averages
- Join averages to society polygons and render a choropleth + popups


In [ ]:
import requests
import json
import geopandas as gpd
# !pip install osm2geojson
import osm2geojson

# --- 1. CONFIGURATION ---
# The Overpass API endpoint
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Bounding box for the Lahore area [S, W, N, E]
# Same as the one used in the HTML file
LAHORE_BOUNDS = "31.3,74.1,31.7,74.6"

# Overpass QL queries
# We use the same queries from the web map to ensure we get the same data.
QUERIES = {
    "union_councils": {
        "query": f"""
            [out:json][timeout:30];
            (relation["boundary"="administrative"]["admin_level"="8"]({LAHORE_BOUNDS}););
            out body;
            >;
            out skel qt;
        """,
        "filename": "lahore_union_councils"
    },
    "housing_societies": {
        "query": f"""
            [out:json][timeout:60];
            (
              way["landuse"="residential"]["name"]({LAHORE_BOUNDS});
              relation["landuse"="residential"]["name"]({LAHORE_BOUNDS});
            );
            out body;
            >;
            out skel qt;
        """,
        "filename": "lahore_housing_societies"
    }
}

# --- 2. CORE FUNCTIONS ---

def fetch_osm_data(query_string: str) -> dict:
    """
    Sends a query to the Overpass API and returns the JSON response.
    """
    print("Sending query to Overpass API...")
    try:
        response = requests.get(OVERPASS_URL, params={'data': query_string})
        response.raise_for_status()  # Raises an HTTPError for bad responses (4xx or 5xx)
        print("Data received successfully.")
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data: {e}")
        return None

def save_as_geojson(geojson_data: dict, filename: str):
    """
    Saves a GeoJSON dictionary to a .geojson file.
    """
    filepath = f"{filename}.geojson"
    try:
        with open(filepath, 'w') as f:
            json.dump(geojson_data, f, indent=2)
        print(f"Successfully saved data to {filepath}")
    except IOError as e:
        print(f"Error saving GeoJSON file: {e}")

def save_as_shapefile(geojson_data: dict, filename: str):
    """
    Converts GeoJSON data to a GeoDataFrame and saves it as a Shapefile.
    """
    filepath = f"{filename}.shp"
    print(f"Processing data for Shapefile conversion...")
    
    # Check if there are features to process
    if not geojson_data.get('features'):
        print("No features found in GeoJSON data. Skipping Shapefile creation.")
        return
        
    try:
        # GeoPandas can directly read GeoJSON features
        gdf = gpd.GeoDataFrame.from_features(geojson_data["features"])

        # Set the coordinate reference system (CRS) to WGS84
        gdf.crs = "EPSG:4326"

        # The 'tags' column from OSM is a dictionary. Shapefiles have limitations
        # on field types, so we'll expand the most important tag ('name') into its own column.
        if 'tags' in gdf.columns:
            gdf['name'] = gdf['tags'].apply(lambda tags: tags.get('name') if isinstance(tags, dict) else None)
            # You could expand other tags here if needed
            # e.g., gdf['landuse'] = gdf['tags'].apply(lambda x: x.get('landuse'))
        
        # Select columns that are simple types (string, number) for the shapefile.
        # We'll keep the geometry and the name.
        columns_to_keep = ['geometry', 'name']
        # Filter the gdf to only include columns that actually exist
        final_columns = [col for col in columns_to_keep if col in gdf.columns]
        
        gdf_simplified = gdf[final_columns]
        
        gdf_simplified.to_file(filepath, driver='ESRI Shapefile')
        print(f"Successfully saved data to {filepath}")
    except Exception as e:
        print(f"An error occurred while creating the Shapefile: {e}")
        print("Note: Ensure all geometries are valid. Sometimes complex OSM relations can cause issues.")


# --- 3. MAIN EXECUTION ---

if __name__ == "__main__":
    print("--- Starting OSM Data Download Script ---")
    
    for key, value in QUERIES.items():
        print(f"\nProcessing query for: {key.replace('_', ' ').title()}")
        
        osm_data = fetch_osm_data(value["query"])
        
        if osm_data:
            # Convert raw OSM JSON to GeoJSON format
            geojson_data = osm2geojson.json2geojson(osm_data)
            
            # Save the data in both formats
            save_as_geojson(geojson_data, value["filename"])
            save_as_shapefile(geojson_data, value["filename"])
            
    print("\n--- Script finished. ---")


--- Starting OSM Data Download Script ---

Processing query for: Union Councils
Sending query to Overpass API...
Data received successfully.
Successfully saved data to lahore_union_councils.geojson
Processing data for Shapefile conversion...
No features found in GeoJSON data. Skipping Shapefile creation.

Processing query for: Housing Societies
Sending query to Overpass API...
Data received successfully.
Successfully saved data to lahore_housing_societies.geojson
Processing data for Shapefile conversion...
Successfully saved data to lahore_housing_societies.shp

--- Script finished. ---


In [15]:
import requests
import json
import geopandas as gpd
import osm2geojson
import folium
import os

# --- 1. CONFIGURATION ---
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
# A tighter bounding box for the initial search for Lahore's boundary
LAHORE_BOUNDS = "31.35,74.15,31.75,74.55"
LAHORE_CENTER = [31.5204, 74.3587]

# Query to get the administrative boundary for Lahore District
LAHORE_BOUNDARY_QUERY = f"""
[out:json][timeout:30];
rel["name"="Lahore"]["admin_level"="5"]({LAHORE_BOUNDS});
out geom;
"""

# Query for Union Councils remains the same
UC_QUERY = f"""
[out:json][timeout:30];
(relation["boundary"="administrative"]["admin_level"="8"]({LAHORE_BOUNDS}););
out body;
>;
out skel qt;
"""

# Dictionary to manage file saving and map styling
LAYERS_CONFIG = {
    "lahore_boundary": {
        "query": LAHORE_BOUNDARY_QUERY,
        "filename": "lahore_boundary",
        "style": {"color": "#e32e22", "weight": 4, "opacity": 0.9, "fillOpacity": 0.0}
    },
    "union_councils": {
        "query": UC_QUERY,
        "filename": "lahore_union_councils",
        "style": {"color": "#ff7800", "weight": 3, "opacity": 0.8}
    },
    "housing_societies": {
        # The query will be constructed dynamically after getting Lahore's boundary ID
        "query": "", 
        "filename": "lahore_housing_societies",
        "style": {"color": "#4682B4", "weight": 2, "opacity": 0.7, "fillOpacity": 0.2}
    }
}

# --- 2. CORE LOGIC ---

def download_and_save_data():
    """
    Fetches Lahore's boundary first, then uses it to find societies within it.
    Also fetches Union Councils.
    """
    lahore_area_id = None
    
    # 1. Fetch Lahore Boundary to use as a filter
    key = "lahore_boundary"
    params = LAYERS_CONFIG[key]
    print(f"--- Processing: {key.replace('_', ' ').title()} ---")
    filepath = f"{params['filename']}.geojson"
    try:
        response = requests.get(OVERPASS_URL, params={'data': params['query']})
        response.raise_for_status()
        osm_data = response.json()
        
        # To filter by an area in Overpass, the relation ID must be increased by 3600000000.
        if osm_data.get('elements'):
            lahore_relation_id = osm_data['elements'][0]['id']
            lahore_area_id = lahore_relation_id + 3600000000
            print(f"Found Lahore relation ID: {lahore_relation_id} -> Area ID for filtering: {lahore_area_id}")

        geojson_data = osm2geojson.json2geojson(osm_data)
        with open(filepath, 'w') as f:
            json.dump(geojson_data, f)
        print(f"Successfully saved to {filepath}\n")
    except Exception as e:
        print(f"Could not fetch Lahore's boundary. Aborting societies search. Error: {e}\n")
        # Clear the housing societies file if it exists, as it's now outdated
        if os.path.exists(LAYERS_CONFIG["housing_societies"]["filename"] + ".geojson"):
            os.remove(LAYERS_CONFIG["housing_societies"]["filename"] + ".geojson")
        lahore_area_id = None # Ensure we don't proceed

    # 2. Fetch Housing Societies ONLY if we found the Lahore boundary
    if lahore_area_id:
        key = "housing_societies"
        params = LAYERS_CONFIG[key]
        
        # This query finds societies within the specific area ID of Lahore
        societies_query = f"""
        [out:json][timeout:90];
        area({lahore_area_id})->.searchArea;
        (
          way["landuse"="residential"]["name"](area.searchArea);
          relation["landuse"="residential"]["name"](area.searchArea);
        );
        out body;
        >;
        out skel qt;
        """
        
        print(f"--- Processing: {key.replace('_', ' ').title()} (filtered by Lahore boundary) ---")
        filepath = f"{params['filename']}.geojson"
        try:
            response = requests.get(OVERPASS_URL, params={'data': societies_query})
            response.raise_for_status()
            geojson_data = osm2geojson.json2geojson(response.json())
            with open(filepath, 'w') as f:
                json.dump(geojson_data, f)
            print(f"Successfully saved to {filepath}\n")
        except Exception as e:
            print(f"Error fetching housing societies: {e}\n")

    # 3. Fetch Union Councils (this query is independent)
    key = "union_councils"
    params = LAYERS_CONFIG[key]
    print(f"--- Processing: {key.replace('_', ' ').title()} ---")
    filepath = f"{params['filename']}.geojson"
    try:
        response = requests.get(OVERPASS_URL, params={'data': params['query']})
        response.raise_for_status()
        geojson_data = osm2geojson.json2geojson(response.json())
        with open(filepath, 'w') as f:
            json.dump(geojson_data, f)
        print(f"Successfully saved to {filepath}\n")
    except Exception as e:
        print(f"Error fetching Union Councils: {e}\n")

def create_map():
    """Creates a Folium map and plots the saved GeoJSON files with clickable popups."""
    print("--- Creating Interactive Map ---")
    m = folium.Map(location=LAHORE_CENTER, zoom_start=11, tiles="CartoDB positron")

    for key, params in LAYERS_CONFIG.items():
        filepath = f"{params['filename']}.geojson"
        if os.path.exists(filepath):
            try:
                print(f"Adding layer: {filepath}")
                
                # Use GeoPandas for robust data loading and cleaning
                gdf = gpd.read_file(filepath)
                if gdf.empty:
                    print(f"Warning: {filepath} is empty. Skipping.")
                    continue

                # Safely parse the 'tags' column to extract the 'name' for the popup
                def parse_tags_for_name(tags_cell):
                    if tags_cell is None: return 'N/A'
                    try:
                        # Geopandas can read the tags dict as a string, so we use eval to parse it.
                        tags_dict = eval(str(tags_cell))
                        if isinstance(tags_dict, dict):
                            return tags_dict.get('name', 'N/A')
                    except Exception:
                        return 'N/A' # Return default if parsing fails
                    return 'N/A'

                # Create a clean 'name' column for the popup
                if 'tags' in gdf.columns:
                    gdf['name'] = gdf['tags'].apply(parse_tags_for_name)
                else:
                    gdf['name'] = 'N/A'
                
                # Keep only the essential columns for Folium
                gdf_cleaned = gdf[['geometry', 'name']]

                # Add the cleaned data to the map with a clickable popup
                gjson = folium.GeoJson(
                    gdf_cleaned,
                    name=key.replace('_', ' ').title(),
                    style_function=lambda x, style=params['style']: style,
                    popup=folium.GeoJsonPopup(
                        fields=['name'],
                        aliases=['Name:'], 
                        localize=True,
                        sticky=False
                    )
                ).add_to(m)

            except Exception as e:
                print(f"Could not add layer from {filepath}: {e}")

    folium.LayerControl().add_to(m)
    map_filename = "lahore_interactive_map.html"
    m.save(map_filename)
    print(f"\nMap saved successfully to {map_filename}")

# --- 3. MAIN EXECUTION ---
if __name__ == "__main__":
    download_and_save_data()
    create_map()
    print("\n--- Script finished. ---")



--- Processing: Lahore Boundary ---
Successfully saved to lahore_boundary.geojson

--- Processing: Union Councils ---


Skipping field nodes: unsupported OGR type: 13


Successfully saved to lahore_union_councils.geojson

--- Creating Interactive Map ---
Adding layer: lahore_boundary.geojson
Adding layer: lahore_union_councils.geojson
Adding layer: lahore_housing_societies.geojson

Map saved successfully to lahore_interactive_map.html

--- Script finished. ---
